Импортирую данные, настраиваю рандомное поведение np.random.

In [1]:
import pandas as pd
import numpy as np
from catboost import Pool, CatBoostRegressor

np.random.seed(42)

Функция предобработки таблицы:
- date заменяет на два столбца типа int, year и month
- Заполняет пропущенные значения metro_dist (находит среднее по категории)
- Заполняет пропущенные данные build_tech и g_lift (случайное значение из распределения известных значений этих данных)

In [2]:
def preprocess_df(df):
    df = df.drop("id", axis=1)

    # date
    date = np.array([i.split('-') for i in df["date"]])
    df = df.drop("date", axis=1)
    df.insert(1, "month", np.array([int(i[1]) for i in date]))
    df.insert(1, "year", np.array([int(i[0]) for i in date]))

    # metro_dist
    mask = np.isnan(df["metro_dist"])
    metro_dist_mean = df["metro_dist"][~mask].mean().round(2)
    for cat in np.unique(df.loc[mask, "street_id"]):
        metro_dist_mean_cat = df.loc[(df["street_id"] == cat) & (
            ~mask), "metro_dist"].mean().round(2)
        metro_dist_mean_cat = metro_dist_mean if np.isnan(
            metro_dist_mean_cat) else metro_dist_mean_cat

        mask_cat = (df["street_id"] == cat) & mask
        df.loc[mask_cat, "metro_dist"] = metro_dist_mean_cat

    # build_tech
    mask = np.isnan(df["build_tech"])
    build_tech = df[~mask]["build_tech"]
    df.loc[mask, "build_tech"] = np.random.choice(
        build_tech, size=len(df[mask]), replace=True)
    df["build_tech"] = df["build_tech"].astype("int64")

    # g_lift
    mask = np.isnan(df["g_lift"])
    g_lift = df[~mask]["g_lift"]
    df.loc[mask, "g_lift"] = np.random.choice(
        g_lift, size=len(df[mask]), replace=True)

    return df

Использую модель CatBoostRegressor, т.к она способна автоматически обрабатывать категориальные признаки. Их я находил перебором.
loss_function задаю как MAE т.к по заданию проверяется эта метрика.
Остальные параметры CatBoostRegressor также находил перебором.

In [3]:
df_train = pd.read_csv("Train.csv")
df_train = preprocess_df(df_train)

X, y = df_train.iloc[:, :-1].copy(), df_train["price"].to_numpy()

cat_features = ["street_id", "build_tech"]
data = Pool(data=X, label=y, cat_features=cat_features)

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.1,
    depth=10,
    loss_function="MAE")
model.fit(data, verbose=False)

Получение предсказания.

In [4]:
# test
df_test = pd.read_csv("Test.csv")
df_test = preprocess_df(df_test)

X_test = df_test.copy()
y_pred = model.predict(X_test).round(2)

Запись данных в submission.csv.

In [5]:
# save_data
data = {
    "id": np.arange(100000, 199999 + 1),
    "price": y_pred
}

df = pd.DataFrame(data)
df.to_csv("submission.csv", index=False)